# Herning — envelope model check

Checks the multiplicative model for AE and UL discussed on 2026-10-29. Per frequency band $b$,

$$x_b(t) = \sigma_b(t)\,g_b(t),$$

with $g_b$ a stationary narrowband noise (the carrier) and $\sigma_b \ge 0$ a slowly varying intensity.
The band energy averaged over blocks of length $T$ is then $E_b[n] \propto \sigma_b^2(nT)\,\chi_b[n]$, and

$$L_b[n] = \ln E_b[n] = \ln\sigma_b^2(nT) + \ln\chi_b[n].$$

If the carrier is Gaussian, $\chi_b$ is approximately gamma distributed with a shape $a_b$ that follows from the
band's own spectrum and $T$ (about $B_b T$ for wide bands, about 1 for bands narrower than $1/T$). So the
fluctuation of $L_b$ that the carrier alone produces is known — the **null** — and anything beyond it is a
change of intensity (rotation-locked modulation or bursts).

Three checks:
1. **Variance.** Noise reference (bearing not rotating): does $L_b$ fluctuate as the null predicts? Rotating: how much
   variance is there beyond the null, per band?
2. **Distribution.** How is the excess distributed? A heavy upper tail means bursts; a symmetric widening means smooth modulation.
3. **Shared intensity.** Do the bands (AE and UL together) move together, i.e. is there one common $\sigma(t)$?

Not part of any pipeline and decides nothing (D68). Settings approved for this notebook (2026-10-29): bands as the feature
baseline (D60/D61), blocks of 125 samples = 20 kHz (as `herning_eda.ipynb` section 8), sweeps = section 5 picks plus the
noise reference at 40/60/100 °C (five each), residual = $L_b$ minus the band level only. Added on request (2026-10-29): the
sweeps with the VFD off (rpm target 0) as a second reference, and the burst rate over the whole run against κ (section 8).
Other choices made only for looking are marked *EDA choice*.

Added on request (2026-10-30): a scan over block lengths that chooses $T$ from the data instead of fixing 50 µs (section 9),
and a test of the multiplicative structure itself: do loud and quiet stretches of a band have the same spectral shape (section 10)?
Added on request (2026-10-30): the same test with each band split into narrower sub-bands (section 11), to find the band
width at which one carrier per band describes the data.

**Run:** `uv sync --group eda`, then open this notebook. Sections 4–7 read 51 sweeps (about a minute); section 8 reads
every 4th sweep of the run (about 15 minutes); sections 9–11 about a minute each.

Sections: 1. Setup  2. Sweeps  3. Bands  4. Band energies  5. Check 1: variance  6. Check 2: distribution
7. Check 3: shared intensity  8. Burst rate over the run  9. Scale scan  10. Same shape  11. Same shape at finer band widths
12. Reading the results

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.signal import spectrogram
from scipy.special import polygamma
from scipy.stats import loggamma, spearmanr

from harness import kappa
from harness.data import herning


def repo_root():
    """The repository root: the first folder upwards that contains pyproject.toml."""
    p = Path.cwd().resolve()
    while not (p / "pyproject.toml").exists():
        p = p.parent
    return p


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "eda" / "herning"))
from check_bands import role  # sweep roles exactly as in the band check (D52, D55)

BAND_CFG = yaml.safe_load(open(ROOT / "configs" / "band_check_herning.yaml"))
OP = BAND_CFG["operating_point"]  # D55
f = h5py.File(ROOT / BAND_CFG["source"], "r")
SWEEPS = f["sweeps"]
FS = 2.5e6  # scope sample rate, both channels

# Plot colours as in herning_eda.ipynb.
COLOR = {"AE": "#2a78d6", "UL": "#eb6834"}
MUTED = "#52514e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

## 2. Sweeps
Sweep table from the telemetry, as in `herning_eda.ipynb` section 3 (rpm and temperature as D55, roles and κ as the
band check). Groups: the section 5 picks plus two references where the bearing does not rotate: the noise reference
(rpm target 100: VFD on) and the VFD switched off (rpm target 0). Comparing the two separates the converter from the rest.

In [ ]:
rows = []
for name in herning.sweep_names(f):  # time order
    a = SWEEPS[name].attrs
    rpm = float(a["telem_vfd_cmd_hz"]) * OP["rpm_per_cmd_hz"]
    temp = float(a[OP["temperature_field"]])
    r = role(float(a["telem_rpm_target"]), float(a["telem_temp_target"]), rpm, temp, OP)
    rows.append({"sweep": name, "rpm_target": float(a["telem_rpm_target"]), "temp_target": float(a["telem_temp_target"]),
                 "rpm": rpm, "temperature_c": temp, "role": r,
                 "kappa": kappa.kappa(rpm, temp, OP["d_pw_mm"], OP["nu_40_cst"], OP["nu_100_cst"]) if r == "analysed" else np.nan})
sweeps = pd.DataFrame(rows).set_index("sweep")


def describe(name):
    r = sweeps.loc[name]
    return f"{name}: {r.rpm:.0f} rpm, {r.temperature_c:.0f} °C, κ={r.kappa:.3f}, {r.role}"


# Groups as (rpm target, temperature target), approved 2026-10-29.
GROUPS = {"off 40 °C": (0, 40), "off 60 °C": (0, 60), "off 100 °C": (0, 100),  # VFD off (added 2026-10-29)
          "ref 40 °C": (100, 40), "ref 60 °C": (100, 60), "ref 100 °C": (100, 100),
          "500 rpm 60 °C": (500, 60), "1500 rpm 60 °C": (1500, 60), "3000 rpm 60 °C": (3000, 60),
          "1500 rpm 40 °C": (1500, 40), "1500 rpm 100 °C": (1500, 100)}
N_PER_GROUP = 5
FIRST = 2  # EDA choice: start at the third sweep of each (rpm, temperature) target, as section 5 (k=2)

picks = {}
for g, (r, t) in GROUPS.items():
    names = sweeps.index[(sweeps.rpm_target == r) & (sweeps.temp_target == t)]
    picks[g] = list(names[FIRST:FIRST + N_PER_GROUP])
    print(f"{g:16s} {len(picks[g])} of {len(names)} sweeps, roles: {', '.join(sweeps.loc[picks[g], 'role'].unique())}")

# Colour and marker per group in the figures: VFD off green, noise reference grey, rotating blue (in the order above).
STYLE = {}
for prefix, cmap, marker in [("off", "Greens", "^-"), ("ref", "Greys", "o-"), ("", "Blues", "s-")]:
    gs = [g for g in GROUPS if g not in STYLE and g.startswith(prefix)]
    for i, g in enumerate(gs):
        STYLE[g] = (plt.get_cmap(cmap)(0.4 + 0.6 * i / max(len(gs) - 1, 1)), marker)

## 3. Bands
Bands as the feature baseline: AE octaves 50–100–200–400–560 kHz (D61); UL base-10 octaves with centres 1.0 … 126 kHz,
edges at centre·10^(±0.15) (D60).

Each band is cut out of the sweep's FFT with a mask whose edges are short cosine transitions. At every inner edge the
falling cos of the lower band and the rising sin of the upper band have squares that add to 1 (power complementary), so
the band powers of a channel add up to the channel's power between the outer edges. The table gives each band's width $B$,
the carrier's correlation time $1/B$, and $B\cdot T$ for $T$ = 50 µs.

In [ ]:
BLOCK = 125  # samples per block: T = 50 us, envelope rate 20 kHz (approved 2026-10-29, as herning_eda section 8)
T = BLOCK / FS
TRANSITION = 0.1  # EDA choice: width of the cosine transition at each band edge, as a fraction of the edge frequency

EDGES = {"AE": [50e3, 100e3, 200e3, 400e3, 560e3],                     # D61
         "UL": [1e3 * 10 ** (0.3 * k - 0.15) for k in range(9)]}       # D60: 0.71, 1.41, ..., 178 kHz
BANDS = [(ch, lo, hi) for ch in EDGES for lo, hi in zip(EDGES[ch][:-1], EDGES[ch][1:])]
LABELS = [f"{ch} {lo / 1e3:.3g}–{hi / 1e3:.3g}" for ch, lo, hi in BANDS]  # kHz


def masks(freqs, edges, w=TRANSITION):
    """Amplitude responses (on the grid `freqs`) of the contiguous bands between `edges`.
    Around an edge e the response rises (sin) or falls (cos) over [e(1 - w/2), e(1 + w/2)]."""
    def s(e):  # 0 below the transition, 1 above, linear in between
        return np.clip((freqs - e * (1 - w / 2)) / (e * w), 0, 1)
    return [np.sin(np.pi / 2 * s(lo)) * np.cos(np.pi / 2 * s(hi)) for lo, hi in zip(edges[:-1], edges[1:])]


N = len(SWEEPS[picks["ref 60 °C"][0]]["AE"]["voltage"])  # samples per sweep (500,000)
FREQS = np.fft.rfftfreq(N, 1 / FS)
MASKS = {ch: masks(FREQS, EDGES[ch]) for ch in EDGES}

pd.DataFrame({"band (kHz)": LABELS, "B (kHz)": [(hi - lo) / 1e3 for _, lo, hi in BANDS],
              "1/B (µs)": [1e6 / (hi - lo) for _, lo, hi in BANDS],
              "B·T": [(hi - lo) * T for _, lo, hi in BANDS]}).set_index("band (kHz)").round(3)

In [ ]:
# The squared masks per channel and their sum (1 between the outer edges, apart from the outer transitions).
fig, ax = plt.subplots(1, 2, figsize=(12, 2.8))
for a, ch in zip(ax, EDGES):
    for m in MASKS[ch]:
        a.semilogx(FREQS[1:], m[1:] ** 2, lw=0.8, color=COLOR[ch])
    a.semilogx(FREQS[1:], sum(m ** 2 for m in MASKS[ch])[1:], lw=1, ls="--", color=MUTED, label="sum of squares")
    a.set_xlim(300, 1e6); a.set_xlabel("frequency (Hz)"); a.set_ylabel(f"{ch}: squared response"); a.legend(frameon=False)
plt.tight_layout()

## 4. Band energies
For every band: the analytic band signal $z_b$ (the sweep's positive-frequency spectrum times the mask, doubled, inverse FFT),
its instantaneous power $p_b = |z_b|^2/2$ (whose mean is the band's power), and the block means $E_b[n]$ of $p_b$ over 125 samples.

**Null.** For a stationary Gaussian carrier with the band's spectrum, $E_b$ is approximately gamma distributed (moments matched)
with shape $a_b = 1/r_b$, where $r_b = \mathrm{Var}(E_b)/\mathrm{E}(E_b)^2 = \frac{1}{M}\sum_{|j|<M}(1-|j|/M)\,|\rho_b(j)|^2$,
$M$ = 125 and $\rho_b$ the autocorrelation of $z_b$. $\rho_b$ is taken from the sweep's own band spectrum, so the null uses the
carrier's actual spectral shape (lines included) and nothing else. Then $\mathrm{Var}(\ln E_b) = \psi'(a_b)$ (trigamma).

In [ ]:
def analytic_bands(name, max_lag):
    """For every band of one sweep (in BANDS order): the analytic band signal z and |rho(j)|^2 for lags j = 0 ... max_lag-1."""
    for ch in EDGES:
        v = SWEEPS[name][ch]["voltage"][:]
        X = np.fft.rfft(v - v.mean())
        for m in MASKS[ch]:
            Z = np.zeros(N, complex)
            Z[: len(X)] = 2 * X * m          # positive frequencies only (the masks are 0 at DC and Nyquist)
            R = np.fft.ifft(np.abs(Z) ** 2)  # circular autocorrelation of z (unnormalised)
            yield np.fft.ifft(Z), np.abs(R[:max_lag] / R[0]) ** 2


def null_shape(rho2, M):
    """Gamma shape a = 1 / r of block means of |z|^2 over M samples under the null; |rho(-j)| = |rho(j)|."""
    j = np.arange(1, M)
    return M / (1 + 2 * np.sum((1 - j / M) * rho2[1:M]))


def band_energies(name):
    """Block energies E (bands x blocks, V^2) of one sweep and the null's gamma shape a (bands)."""
    E, a = [], []
    for z, rho2 in analytic_bands(name, BLOCK):
        p = np.abs(z) ** 2 / 2  # instantaneous power
        E.append(p[: N // BLOCK * BLOCK].reshape(-1, BLOCK).mean(axis=1))
        a.append(null_shape(rho2, BLOCK))
    return np.array(E), np.array(a)


results = {}  # sweep name -> group, L = ln E (bands x blocks), a (bands)
for g, names in picks.items():
    for name in names:
        E, a = band_energies(name)
        results[name] = {"group": g, "L": np.log(E), "a": a}
print(len(results), "sweeps;", results[name]["L"].shape[1], "blocks each")

In [ ]:
# One noise-reference sweep and one 3000 rpm sweep: L_b minus its sweep mean, all bands (rows) over the sweep.
# Dotted lines (3000 rpm): one per revolution, arbitrary phase (no shaft reference in the data).
SHOW = [picks["ref 60 °C"][0], picks["3000 rpm 60 °C"][0]]  # EDA choice
t0 = float(SWEEPS[SHOW[0]]["AE"].attrs["x_origin"])  # sweeps start at about -100 ms
fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
for a_, name in zip(ax, SHOW):
    L = results[name]["L"]
    t_ms = 1e3 * (t0 + (np.arange(L.shape[1]) + 0.5) * T)
    im = a_.pcolormesh(t_ms, np.arange(len(BANDS)), L - L.mean(axis=1, keepdims=True), cmap="RdBu_r",
                       vmin=-3, vmax=3, shading="nearest")  # EDA choice: colour range +-3 (ln units)
    a_.set_yticks(range(len(BANDS)), LABELS, fontsize=7)
    a_.set_title(describe(name), loc="left", fontsize=9)
    plt.colorbar(im, ax=a_, label="ln E − sweep mean")
rpm = sweeps.loc[SHOW[1], "rpm"]
for x in np.arange(t_ms[0], t_ms[-1], 60e3 / rpm):
    ax[1].axvline(x, color="k", lw=0.5, ls=":")
ax[-1].set_xlabel("time (ms)")
plt.tight_layout()

In [ ]:
# The same two sweeps, two bands as lines (EDA choice: which bands).
LINE_BANDS = ["AE 100–200", "UL 11.2–22.4"]
fig, ax = plt.subplots(len(LINE_BANDS), 1, figsize=(12, 4.5), sharex=True)
for a_, lab in zip(ax, LINE_BANDS):
    b = LABELS.index(lab)
    for name, c in zip(SHOW, [MUTED, COLOR[lab[:2]]]):
        L = results[name]["L"][b]
        a_.plot(t_ms, L - L.mean(), lw=0.5, color=c, label=describe(name).split(": ")[1].split(", κ")[0])
    a_.set_ylabel(f"{lab} kHz\nln E − mean")
ax[0].legend(frameon=False, fontsize=7); ax[-1].set_xlabel("time (ms)")
plt.tight_layout()

## 5. Check 1: variance beyond the null
Per sweep and band: the observed variance of $L_b$ over the sweep divided by the null variance $\psi'(a_b)$. Table and figure:
median over the five sweeps of a group.
- ratio ≈ 1: the band fluctuates exactly as a stationary Gaussian carrier would;
- ratio > 1: intensity changes (modulation, bursts). $1 - 1/\text{ratio}$ is the share of $\mathrm{Var}(L_b)$ that is intensity;
- ratio < 1: less fluctuation than Gaussian noise, e.g. deterministic lines (a sinusoid has a constant envelope; VFD comb, QUESTIONS.md Q3).

In [ ]:
rows = []
for name, r in results.items():
    for b, lab in enumerate(LABELS):
        null_var = polygamma(1, r["a"][b])  # Var(ln E) under the null: trigamma(a)
        rows.append({"group": r["group"], "sweep": name, "band": lab, "a": r["a"][b],
                     "null_var": null_var, "obs_var": r["L"][b].var(), "ratio": r["L"][b].var() / null_var})
check1 = pd.DataFrame(rows)

shape = check1.groupby("band").a.median().reindex(LABELS).rename("null shape a (median)")
ratio = check1.groupby(["band", "group"]).ratio.median().unstack("group").reindex(index=LABELS, columns=list(GROUPS))
pd.concat([shape, ratio], axis=1).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
for g in GROUPS:
    c, mk = STYLE[g]
    ax.semilogy(range(len(LABELS)), ratio[g], mk, ms=4, lw=1, color=c, label=g)
ax.axhline(1, color=MUTED, lw=0.8, ls="--")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Var(ln E) / null variance"); ax.legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout()

## 6. Check 2: distribution of the residual
Residual = $L_b$ minus the band level only (approved 2026-10-29), pooled over the five sweeps of a group. Each sweep is
aligned by its **median** (robust to bursts) to the null's median. Grey curve: the null log-gamma density for the
group's median shape $a_b$. Log density axis, so the tails are visible.

In [ ]:
HIST_BANDS = ["AE 100–200", "AE 200–400", "UL 11.2–22.4", "UL 44.7–89.1"]  # EDA choice
HIST_GROUPS = ["off 60 °C", "ref 60 °C", "500 rpm 60 °C", "3000 rpm 60 °C"]  # EDA choice


def residual(name, b):
    """L_b of one sweep, shifted so that its median equals the null's median (null: ln of a gamma(a) variable)."""
    L, a = results[name]["L"][b], results[name]["a"][b]
    return L - np.median(L) + loggamma(a).median()


fig, ax = plt.subplots(1, len(HIST_BANDS), figsize=(13, 3.4), sharey=True)
for a_, lab in zip(ax, HIST_BANDS):
    b = LABELS.index(lab)
    for g in HIST_GROUPS:
        c = STYLE[g][0]
        y = np.concatenate([residual(n, b) for n in picks[g]])
        a_.hist(y, bins=80, density=True, histtype="step", lw=1, color=c, label=g)
    a_med = np.median([results[n]["a"][b] for g in HIST_GROUPS for n in picks[g]])
    grid = np.linspace(*a_.get_xlim(), 400)
    a_.plot(grid, loggamma(a_med).pdf(grid), color="k", lw=0.8, ls="--", label=f"null, a={a_med:.2f}")
    a_.set_yscale("log"); a_.set_ylim(1e-4, None); a_.set_title(f"{lab} kHz", loc="left", fontsize=9)
    a_.set_xlabel("ln E (aligned)")
ax[0].set_ylabel("density"); ax[-1].legend(frameon=False, fontsize=7)
plt.tight_layout()

In [ ]:
# Upper tail: share of blocks above the null's 99.9 % quantile (0.001 under the null), mean over a group's sweeps.
TAIL_Q = 0.999  # EDA choice
tail = {}
for g, names in picks.items():
    tail[g] = [np.mean([np.mean(residual(n, b) > loggamma(results[n]["a"][b]).ppf(TAIL_Q)) for n in names])
               for b in range(len(LABELS))]
pd.DataFrame(tail, index=LABELS).round(4)

## 7. Check 3: shared intensity across bands
Per sweep: the correlation matrix of $L_b$ across the 12 bands (AE and UL together, same clock). Under the null the bands are
nearly independent (disjoint frequency bands of a stationary Gaussian process), so the matrix is close to the identity and all
eigenvalues are close to 1. A common intensity $\sigma(t)$ gives positive correlations everywhere and one large eigenvalue.
Correlations of noisy bands (small $a_b$) are attenuated: by $\sqrt{s_b s_c}$, where $s_b = 1 - 1/\text{ratio}$ is the intensity
share from Check 1.

In [ ]:
CORR_GROUPS = ["off 60 °C", "ref 60 °C", "500 rpm 60 °C", "3000 rpm 60 °C"]  # EDA choice
fig, ax = plt.subplots(1, len(CORR_GROUPS), figsize=(16, 4))
for a_, g in zip(ax, CORR_GROUPS):
    C = np.mean([np.corrcoef(results[n]["L"]) for n in picks[g]], axis=0)  # mean over the group's sweeps
    im = a_.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
    a_.set_xticks(range(len(LABELS)), LABELS, rotation=90, fontsize=6)
    a_.set_yticks(range(len(LABELS)), LABELS, fontsize=6)
    a_.set_title(g, loc="left", fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8, label="correlation of ln E")

In [ ]:
# Eigenvalues of each sweep's correlation matrix (largest first), median per group; and the leading
# eigenvector's loadings per band (sign chosen so the loadings sum to a positive number), median per group.
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for g in GROUPS:
    c, mk = STYLE[g]
    eig, load = [], []
    for n in picks[g]:
        w, V = np.linalg.eigh(np.corrcoef(results[n]["L"]))
        eig.append(w[::-1])
        v1 = V[:, -1]
        load.append(v1 * np.sign(v1.sum()))
    ax[0].plot(range(1, len(LABELS) + 1), np.median(eig, axis=0), mk, ms=3, lw=1, color=c, label=g)
    ax[1].plot(range(len(LABELS)), np.median(load, axis=0), mk, ms=3, lw=1, color=c)
ax[0].axhline(1, color=MUTED, lw=0.8, ls="--"); ax[0].set_xlabel("eigenvalue rank"); ax[0].set_ylabel("eigenvalue")
ax[0].legend(frameon=False, fontsize=7, ncol=2)
ax[1].axhline(0, color=MUTED, lw=0.8); ax[1].set_ylabel("leading eigenvector loading")
ax[1].set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right", fontsize=7)
plt.tight_layout()

In [ ]:
# The two largest eigenvalues per group (median over sweeps). Under the null both are close to 1;
# lambda_1 = 12 would mean all bands move together perfectly.
pd.DataFrame({g: np.median([np.linalg.eigvalsh(np.corrcoef(results[n]["L"]))[::-1][:2] for n in picks[g]], axis=0)
              for g in GROUPS}, index=["lambda_1", "lambda_2"]).T.round(2)

## 8. Burst rate over the whole run
For every `STRIDE`-th sweep of the run (all roles except `excluded`): per band, the tail share (as Check 2: share of blocks
above the null's 99.9 % quantile, 0.001 under the null) and the variance ratio (as Check 1). Then, as `herning_eda.ipynb`
section 10: the tail share against rpm per temperature target, and its Spearman correlation with κ *within* each rpm
level. At a fixed speed κ varies only through temperature, so this shows what tracks κ beyond "which speed is it".

Caveat: within an rpm level, κ, temperature and elapsed time move together (the temperature blocks run one after the
other), so a correlation here cannot separate lubrication from temperature effects on sensors and electronics, or from
slow drift. The two references (bearing not rotating) show how much the statistic follows temperature without the bearing.

In [ ]:
STRIDE = 4  # EDA choice: every 4th sweep, as herning_eda section 10 (STRIDE = 1: all ~3,960 sweeps, about an hour)
subset = sweeps[sweeps.role != "excluded"].iloc[::STRIDE]
rows = []
for i, name in enumerate(subset.index):
    E, a = band_energies(name)
    L = np.log(E)
    for b, lab in enumerate(LABELS):
        y = L[b] - np.median(L[b]) + loggamma(a[b]).median()  # aligned to the null by the median, as residual()
        rows.append({"sweep": name, "band": lab, "tail_share": np.mean(y > loggamma(a[b]).ppf(TAIL_Q)),
                     "ratio": L[b].var() / polygamma(1, a[b])})
    if i % 100 == 0:
        print(f"{i}/{len(subset)}", flush=True)
run = pd.DataFrame(rows).join(sweeps, on="sweep")

In [ ]:
# Tail share against rpm, one line per temperature target (light = cold, dark = hot): median per rpm level.
# Dotted: median of the noise reference (grey) and of the VFD-off sweeps (green), all temperatures.
RUN_BANDS = ["AE 50–100", "AE 200–400", "UL 11.2–22.4", "UL 44.7–89.1"]  # EDA choice
rot = run[run.role == "analysed"]
temps = sorted(rot.temp_target.unique())
fig, ax = plt.subplots(1, len(RUN_BANDS), figsize=(14, 3.4), sharey=True)
for a_, lab in zip(ax, RUN_BANDS):
    for i, t in enumerate(temps):
        m = rot[(rot.band == lab) & (rot.temp_target == t)].groupby("rpm_target").tail_share.median()
        a_.plot(m.index, m.values, lw=0.9, color=plt.get_cmap("Blues")(0.3 + 0.7 * i / (len(temps) - 1)))
    for r, c in [("noise_reference", MUTED), ("non_rotating_0", "#1baf7a")]:
        a_.axhline(run[(run.band == lab) & (run.role == r)].tail_share.median(), color=c, lw=0.9, ls=":")
    a_.axhline(1 - TAIL_Q, color="k", lw=0.6, ls="--")  # the null's value
    a_.set_title(f"{lab} kHz", loc="left", fontsize=9); a_.set_xlabel("rpm")
ax[0].set_ylabel("tail share")
plt.tight_layout()

In [ ]:
# Within-rpm Spearman correlation of tail share and variance ratio with kappa (as herning_eda section 10):
# median over the rpm levels in three speed ranges.
def within_rpm_spearman(df, col):
    out = {}
    for r, g in df.groupby("rpm_target"):
        if len(g) >= 10:
            out[r] = spearmanr(g[col], g.kappa)[0]
    return pd.Series(out)


table = {}
for lab in LABELS:
    for col in ("tail_share", "ratio"):
        s = within_rpm_spearman(rot[rot.band == lab], col)
        table[(col, lab)] = {"≤600 rpm": s[s.index <= 600].median(), "700–1400": s[(s.index > 600) & (s.index < 1500)].median(),
                             "≥1500 rpm": s[s.index >= 1500].median()}
pd.DataFrame(table).T.round(2).unstack(0).swaplevel(axis=1).sort_index(axis=1).reindex(LABELS)

In [ ]:
# The same statistics on the two references against temperature (no bearing rotation): if these are flat,
# a within-rpm trend above is not the electronics following temperature. (Temperature also rises with elapsed time.)
pd.DataFrame({(r, col): {lab: spearmanr(g[col], g.temperature_c)[0] for lab, g in run[run.role == r].groupby("band")}
              for r in ("non_rotating_0", "noise_reference") for col in ("tail_share", "ratio")}).reindex(LABELS).round(2)

## 9. Scale scan: how long can a block be?
The model needs a block length $T$ between two time scales, $1/B_b \ll T \ll \tau_\sigma$ (carrier decorrelation vs. the time
scale on which the intensity changes). Instead of fixing $T$ = 50 µs, the excess variance of Check 1 is computed over a range
of block lengths:

$$\text{excess}(T) = \mathrm{Var}(\ln E_T) - \psi'(a(T)) \approx \mathrm{Var}(\ln \sigma_T^2),$$

the variance of the log intensity after averaging over blocks of length $T$ (if intensity and carrier noise are independent).
Intensity changes faster than $T$ are averaged away, so excess($T$) falls with $T$, and its shape shows the time scales at
which the intensity varies. Read as a trade-off: a longer block means fewer numbers per sweep but less of the intensity
variation kept. The table gives, per band and rotating group, the longest $T$ that keeps at least 90 % and 50 % of the
maximum excess (searching upwards from the maximum).

Calibration: at short $T$ the null is not exact in UL (deterministic lines make bands steadier than the null predicts, so the
excess is negative even on the VFD-off reference). The table therefore subtracts the VFD-off excess at the same temperature,
assuming this bias is the same with and without rotation; the figure shows the uncorrected curves of all groups. Bands with a
small maximum (below about 0.05) hardly vary at any scale, and their $T$ values mean little. At 10 ms a sweep has only
20 blocks, so the longest scales are noisy.

In [ ]:
SCAN = [25, 50, 125, 250, 500, 1250, 2500, 5000, 12500, 25000]  # EDA choice: block lengths in samples (10 µs ... 10 ms)
SCAN_GROUPS = ["off 60 °C", "ref 60 °C", "500 rpm 60 °C", "1500 rpm 60 °C", "3000 rpm 60 °C"]  # EDA choice
RETAIN = [0.9, 0.5]  # EDA choice: fractions of the maximum excess for the table


def scale_scan(name):
    """Excess variance Var(ln E) - psi'(a) of one sweep: bands (rows) x block lengths in SCAN (columns)."""
    out = []
    for z, rho2 in analytic_bands(name, max(SCAN)):
        p = np.abs(z) ** 2 / 2
        row = []
        for M in SCAN:
            E = p[: N // M * M].reshape(-1, M).mean(axis=1)
            row.append(np.log(E).var() - polygamma(1, null_shape(rho2, M)))
        out.append(row)
    return np.array(out)


excess = {g: np.median([scale_scan(n) for n in picks[g]], axis=0) for g in SCAN_GROUPS}  # median over a group's sweeps
T_US = np.array(SCAN) / FS * 1e6

fig, ax = plt.subplots(3, 4, figsize=(14, 8), sharex=True)
for b, (a_, lab) in enumerate(zip(ax.ravel(), LABELS)):
    for g in SCAN_GROUPS:
        c, mk = STYLE[g]
        a_.semilogx(T_US, excess[g][b], mk, ms=3, lw=1, color=c, label=g)
    a_.axhline(0, color=MUTED, lw=0.8, ls="--")
    a_.axvline(1e6 / (BANDS[b][2] - BANDS[b][1]), color=MUTED, lw=0.6, ls=":")  # 1/B_b
    a_.set_title(f"{lab} kHz", loc="left", fontsize=9)
for a_ in ax[-1]:
    a_.set_xlabel("block length T (µs)")
for a_ in ax[:, 0]:
    a_.set_ylabel("excess variance of ln E")
ax[0, 0].legend(frameon=False, fontsize=6)
fig.suptitle("uncorrected; dotted: 1/B (carrier correlation time)", x=0.01, ha="left", fontsize=8)
plt.tight_layout()

In [ ]:
def longest_T(curve, q):
    """Longest block length (µs) up to which the curve stays at or above q times its maximum, searching from the maximum on."""
    k = int(np.argmax(curve))
    while k + 1 < len(curve) and curve[k + 1] >= q * curve.max():
        k += 1
    return T_US[k]


rows = {}
for g in SCAN_GROUPS[2:]:  # the rotating groups
    corrected = excess[g] - excess["off 60 °C"]  # calibration with the VFD-off reference at the same temperature
    rows[(g, "max")] = corrected.max(axis=1)
    for q in RETAIN:
        rows[(g, f"T{int(100 * q)} (µs)")] = [longest_T(c, q) for c in corrected]
pd.DataFrame(rows, index=LABELS).round(3)

## 10. Same shape: are loud stretches scaled copies of quiet ones?
If $x_b = \sigma_b\,g_b$ with one stationary carrier $g_b$, a change of intensity scales the band's spectrum without changing
its shape. So the in-band spectrum of the loudest stretches of a sweep, normalised to unit sum, should match that of the quiet
stretches. If the bursts have their own colour (a second source, or a resonance that moves), the shapes differ, and one
carrier per band is not enough.

Per sweep and band: spectrogram segments, the band energy of each segment, the mean in-band spectrum of the loud segments
(top 10 %) and of the quiet ones (below the median), each normalised; mismatch $D$ = standard deviation over the band's bins
of $10\log_{10}(\text{loud}/\text{quiet})$ in dB (0 = identical shape), and the shift of the in-band spectral centroid from
quiet to loud, as a percentage of the band width (negative: loud stretches carry relatively more energy at the band's low end). Even without any intensity change, $D > 0$ from
estimation noise and from choosing segments by their energy, so the two references (bearing not rotating) give the level
to compare with. Only bands with at least 8 frequency bins at this resolution are tested.
Note: fixed narrow lines (UL, QUESTIONS.md Q3) also raise $D$, because their share of a loud segment is smaller than of a quiet one.

In [ ]:
SHAPE_NPERSEG = 2048  # EDA choice: 0.82 ms segments, 1.2 kHz resolution, half overlap (as herning_eda section 7)
LOUD, QUIET = 0.9, 0.5  # EDA choice: loud = above the 90th percentile of band energy, quiet = below the median
SHAPE_BANDS = [lab for (ch, f_lo, f_hi), lab in zip(BANDS, LABELS) if (f_hi - f_lo) * SHAPE_NPERSEG / FS >= 8]


def shapes(name):
    """Per band in SHAPE_BANDS: (frequencies, loud shape, quiet shape, mismatch D in dB, centroid shift in % of the band width)."""
    out = {}
    for ch in EDGES:
        v = SWEEPS[name][ch]["voltage"][:]
        f_, _, S = spectrogram(v - v.mean(), fs=FS, window="hann", nperseg=SHAPE_NPERSEG, noverlap=SHAPE_NPERSEG // 2)
        for (c, f_lo, f_hi), lab in zip(BANDS, LABELS):
            if c != ch or lab not in SHAPE_BANDS:
                continue
            sel = (f_ >= f_lo) & (f_ < f_hi)
            e = S[sel].sum(axis=0)  # band energy per segment
            loud = S[sel][:, e >= np.quantile(e, LOUD)].mean(axis=1)
            quiet = S[sel][:, e <= np.quantile(e, QUIET)].mean(axis=1)
            loud, quiet = loud / loud.sum(), quiet / quiet.sum()
            shift = 100 * np.sum(f_[sel] * (loud - quiet)) / (f_hi - f_lo)  # centroid(loud) - centroid(quiet), shapes sum to 1
            out[lab] = (f_[sel], loud, quiet, np.std(10 * np.log10(loud / quiet)), shift)
    return out


shape_res = {n: shapes(n) for names in picks.values() for n in names}
# Medians over a group's sweeps: mismatch D (dB) and centroid shift (% of band width).
pd.concat({stat: pd.DataFrame({g: {lab: np.median([shape_res[n][lab][k] for n in picks[g]]) for lab in SHAPE_BANDS}
                               for g in GROUPS}) for stat, k in [("D (dB)", 3), ("shift (%B)", 4)]}).round(2)

In [ ]:
# Loud vs quiet in-band shapes for one VFD-off sweep and one 3000 rpm sweep (EDA choice: the first of each group).
SHAPE_SHOW = [picks["off 60 °C"][0], picks["3000 rpm 60 °C"][0]]
fig, ax = plt.subplots(2, len(SHAPE_BANDS), figsize=(16, 5), sharey=True)
for i, name in enumerate(SHAPE_SHOW):
    for a_, lab in zip(ax[i], SHAPE_BANDS):
        f_, loud, quiet, d, _ = shape_res[name][lab]
        a_.plot(f_ / 1e3, 10 * np.log10(quiet), lw=0.8, color=MUTED, label="quiet")
        a_.plot(f_ / 1e3, 10 * np.log10(loud), lw=0.8, color=COLOR[lab[:2]], label="loud")
        a_.set_title(f"{lab}: D={d:.2f} dB", loc="left", fontsize=7)
    ax[i, 0].set_ylabel(describe(name).split(": ")[1].split(", κ")[0] + "\nshare (dB)", fontsize=7)
for a_ in ax[-1]:
    a_.set_xlabel("kHz")
ax[0, 0].legend(frameon=False, fontsize=6)
plt.tight_layout()

## 11. Same shape at finer band widths
Section 10 found that, in the bands that vary, loud stretches have a darker spectrum than quiet ones, so one carrier per
octave band does not hold. Within a narrow enough band any smooth spectrum is nearly flat, so the background and the
episodes have the same shape locally and one carrier per band holds again; the colour change then appears *between*
neighbouring bands as different intensity series. Here each band is split into $k$ equal-width sub-bands and the test of
section 10 is repeated in every sub-band; $D$ per band and $k$ is the median over its sub-bands. If the model holds at
some width, $D$ on rotating sweeps falls to the references' level there.

Narrow sub-bands need fine frequency resolution to have enough bins, and that needs long spectrogram segments: UL
therefore uses longer segments than AE, so its loud/quiet stretches are judged over 3.3 ms instead of 0.8 ms (the
time–frequency trade-off showing up in the test itself). A sub-band is tested only with at least 8 bins, as in section 10.

In [ ]:
SPLITS = [1, 2, 4, 8, 16]  # EDA choice: each band (D60/D61) split into this many equal-width sub-bands
SPLIT_NPERSEG = {"AE": 2048, "UL": 8192}  # EDA choice: segments of 0.82 ms (1.2 kHz bins) for AE, 3.3 ms (0.31 kHz) for UL
SPLIT_GROUPS = ["off 60 °C", "ref 60 °C", "500 rpm 60 °C", "1500 rpm 60 °C", "3000 rpm 60 °C"]  # EDA choice
MIN_BINS = 8  # as section 10


def shape_D(S, f_, f_lo, f_hi):
    """Mismatch D (dB) of loud vs quiet normalised in-band spectra, as section 10; NaN if the sub-band has too few bins."""
    sel = (f_ >= f_lo) & (f_ < f_hi)
    if sel.sum() < MIN_BINS:
        return np.nan
    e = S[sel].sum(axis=0)  # sub-band energy per segment
    loud = S[sel][:, e >= np.quantile(e, LOUD)].mean(axis=1)
    quiet = S[sel][:, e <= np.quantile(e, QUIET)].mean(axis=1)
    return np.std(10 * np.log10((loud / loud.sum()) / (quiet / quiet.sum())))


def split_shapes(name):
    """{(band, k): median D over the band's k sub-bands} for one sweep (NaN where the sub-bands are too narrow to test)."""
    out = {}
    for ch in EDGES:
        v = SWEEPS[name][ch]["voltage"][:]
        nps = SPLIT_NPERSEG[ch]
        f_, _, S = spectrogram(v - v.mean(), fs=FS, window="hann", nperseg=nps, noverlap=nps // 2)
        for (c, f_lo, f_hi), lab in zip(BANDS, LABELS):
            if c != ch:
                continue
            for k in SPLITS:
                edges = np.linspace(f_lo, f_hi, k + 1)
                d = [shape_D(S, f_, lo, hi) for lo, hi in zip(edges[:-1], edges[1:])]
                out[(lab, k)] = np.nan if np.isnan(d).any() else np.median(d)
    return out


split_res = {n: split_shapes(n) for g in SPLIT_GROUPS for n in picks[g]}
Dk = pd.DataFrame({g: pd.DataFrame([split_res[n] for n in picks[g]]).median() for g in SPLIT_GROUPS})  # median over sweeps
Dk.index = pd.MultiIndex.from_tuples(Dk.index, names=["band", "k"])

fig, ax = plt.subplots(3, 4, figsize=(14, 8))
for a_, (ch, f_lo, f_hi), lab in zip(ax.ravel(), BANDS, LABELS):
    width = (f_hi - f_lo) / np.array(SPLITS) / 1e3  # sub-band width in kHz
    for g in SPLIT_GROUPS:
        c, mk = STYLE[g]
        a_.semilogx(width, Dk.loc[lab, g].reindex(SPLITS).values, mk, ms=3, lw=1, color=c, label=g)
    a_.set_title(f"{lab} kHz", loc="left", fontsize=9)
    a_.invert_xaxis()  # narrower to the right
for a_ in ax[-1]:
    a_.set_xlabel("sub-band width (kHz)")
for a_ in ax[:, 0]:
    a_.set_ylabel("D (dB)")
ax[0, 0].legend(frameon=False, fontsize=6)
plt.tight_layout()

In [ ]:
# D (dB) per band and number of sub-bands k, for the noise reference and the two faster rotating groups (NaN: too narrow to test).
Dk[["ref 60 °C", "1500 rpm 60 °C", "3000 rpm 60 °C"]].unstack("k").reindex(LABELS).round(2)

## 12. Reading the results
- **Check 1 on the noise reference** tests the model itself: ratios near 1 mean the stationary-Gaussian carrier describes the bands
  when nothing rotates, so the null is calibrated. Ratios below 1 point to deterministic lines; above 1 to fluctuations that
  are not the bearing (the bearing does not rotate), e.g. electrical.
- **Check 1 on rotating sweeps**: bands with ratios well above 1 carry intensity changes within a sweep, i.e. information a
  time-resolved envelope has beyond the band level (what the MSSP statistics see). Bands near 1 carry, at this block length,
  only their level.
- **Check 2**: an upper tail above the null while the bulk follows it means sparse bursts; a bulk that is wider on both sides means
  smooth modulation. The tail table gives a burst rate per band (relative to 0.001).
- **Check 3**: one dominant eigenvalue with same-sign loadings over AE and UL bands supports a single common intensity
  $\sigma(t)$ (strong compression possible); several sizeable eigenvalues mean different bands carry different temporal information.
  Structure that also appears in the noise reference is not from the bearing.
- **VFD off vs noise reference**: fluctuation present with the VFD on but not with it off comes from the converter; present in both,
  from something else that does not need the bearing to rotate.
- **Section 8**: a consistent within-rpm correlation of the burst rate with κ, in bands where the references stay flat against
  temperature, is a first sign that the within-sweep dynamics carry lubrication information beyond speed (with the caveat above).
- **Section 9**: a flat excess up to some $T$ means the intensity hardly varies faster than that, so blocks up to that length lose
  little; where it falls, the intensity varies at those time scales. $T_{90}$ and $T_{50}$ are data-based block lengths for a
  chosen loss; compare them across speeds and temperatures (if they move with the operating point, no single $T$ fits).
- **Section 10**: $D$ on rotating sweeps near the references' level means loud stretches are scaled copies of quiet ones, as the
  multiplicative model requires; clearly above it means the loud stretches have their own spectral shape (a second source or a
  moving resonance), and one carrier per band does not describe that band. The centroid shift gives the direction; the shape
  plots show whether the difference is broad or at narrow lines.
- **Section 11**: the sub-band width at which $D$ on rotating sweeps reaches the references' level is the width at which one
  carrier per band describes the data; bands of that width (or narrower) make the multiplicative model usable, at the cost of
  longer blocks ($T \gg 1/B$). If $D$ stays above the references at every testable width, the mismatch is not a smooth colour
  change but something finer (narrow lines or structure below the resolution).
- Not done here: separating the rotation-locked part from bursts (residual = band level only, as approved), other block lengths.